# Sensor data processing pipeline

Flow: **Sensor data → Filtering → Preprocessing → Normalization → Model** (pandas, NumPy, scikit-learn).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from sensor_pipeline import (
    build_feature_pipeline,
    filter_sensor_noise,
    fit_transform_features,
    make_synthetic_labels,
    make_synthetic_sensor_data,
    preprocess_tabular,
)

## 1. Sensor data (tabular, wide format)

One row per timestep; each column is a sensor channel. Replace with `load_sensor_csv("path.csv")` for your own file.

In [ ]:
raw = make_synthetic_sensor_data(n_rows=800, n_sensors=4, seed=42, missing_frac=0.025)
raw = raw.sort_values("timestamp").reset_index(drop=True)
feature_cols = [c for c in raw.columns if c != "timestamp"]
raw.head()

## 2. Filtering (noise reduction)

Rolling median smooths high-frequency noise per channel.

In [ ]:
filtered = filter_sensor_noise(raw, feature_cols, window=7, method="median")
filtered[feature_cols].head()

## 3. Preprocessing (tabular + optional forward-fill)

Numeric coercion and time-oriented `ffill`; remaining gaps are filled in the next step by `SimpleImputer` (fitted on **train only**).

In [ ]:
X_df = preprocess_tabular(filtered, feature_cols, ffill_limit=10)
X_df.head()

## 4. Train / test split (before normalization)

Imputer and scaler statistics come only from the training split to avoid leakage.

In [ ]:
split_idx = int(len(X_df) * 0.8)
X_train, X_test = X_df.iloc[:split_idx], X_df.iloc[split_idx:]
len(X_train), len(X_test)

## 5. Normalization (imputation + scaling)

Sklearn `Pipeline`: `SimpleImputer` → `StandardScaler`.

In [ ]:
feature_pipe = build_feature_pipeline(imputer_strategy="median", scaler="standard")
X_train_n, X_test_n = fit_transform_features(feature_pipe, X_train, X_test)
X_train_n.shape, X_test_n.shape

## 6. AI model (demo classifier)

Synthetic binary labels from sensor_0; a small random forest shows the full path end-to-end.

In [9]:
if "make_synthetic_labels" not in globals():
    from sensor_pipeline import make_synthetic_labels
if "RandomForestClassifier" not in globals():
    from sklearn.ensemble import RandomForestClassifier
if "accuracy_score" not in globals():
    from sklearn.metrics import accuracy_score, classification_report

y = make_synthetic_labels(X_df, feature_cols, seed=7)
y_train, y_test = y[:split_idx], y[split_idx:]

clf = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
clf.fit(X_train_n, y_train)
y_pred = clf.predict(X_test_n)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

ImportError: 

IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!

Importing the numpy C-extensions failed. This error can happen for
many reasons, often due to issues with your setup or how NumPy was
installed.
The following compiled module files exist, but seem incompatible
with with either python 'cpython-312' or the platform 'win32':

  * _multiarray_umath.cp311-win_amd64.lib
  * _multiarray_umath.cp311-win_amd64.pyd

We have compiled some common reasons and troubleshooting tips at:

    https://numpy.org/devdocs/user/troubleshooting-importerror.html

Please note and check the following:

  * The Python version is: Python 3.12 from "c:\cyber security\github\ml-prob\.venv\Scripts\python.exe"
  * The NumPy version is: "2.4.3"

and make sure that they are the versions you expect.

Please carefully study the information and documentation linked above.
This is unlikely to be a NumPy issue but will be caused by a bad install
or environment on your machine.

Original error